# HƯỚNG DẪN KIẾN TRÚC: CHUYỂN DỮ LIỆU TỪ SILVER LÊN GOLD (STAR SCHEMA) & GHI VÀO CLICKHOUSE

Notebook này giải quyết trọn vẹn các vấn đề lớn trong Data Engineering:
1. **Thiết kế bảng Silver với cột `_ingest_at`:** Cột mốc chuẩn hóa để thực hiện **Incremental Load** an toàn lên Gold.
2. **So sánh dbt vs PySpark:** Đưa ra quyết định lựa chọn công nghệ đúng đắn cho tầng Gold.
3. **Xây dựng Star Schema (Fact & Dimension):** Surrogate Key (`md5`), Unknown Member (`UNKNOWN`), Date Key.
4. **Ghi dữ liệu sang ClickHouse qua JDBC:** Thực hiện hành vi tương đương `MERGE INTO` qua Staging Table hoặc `ReplacingMergeTree`.
5. **Quản lý Watermark tầng Gold:** Đảm bảo tính Idempotent khi chạy định kỳ (Airflow/Fabric Scheduler).

## BƯỚC 1: KHỞI TẠO BẢNG TẦNG SILVER VÀ BẢNG WATERMARK CHO GOLD
- **`silver_customer_value`**: Bắt buộc có cột **`_ingest_at`** (thời điểm bản ghi được nạp/cập nhật vào Silver) để phục vụ trích xuất tăng dần lên Gold.
- **`silver_orders`**: Bảng dữ liệu giao dịch sự kiện (Fact nguồn có `_ingested_at`).
- **`ctl_gold_watermark`**: Lưu lại mốc thời gian đã xử lý thành công gần nhất của Gold.

In [ ]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType
from pyspark.sql.functions import col, current_timestamp, lit, expr, md5, date_format

# 1. Khởi tạo bảng Watermark cho tầng Gold
ctl_gold_schema = StructType([
    StructField("pipeline_name", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("last_watermark", TimestampType(), False),
    StructField("last_run_time", TimestampType(), False),
    StructField("status", StringType(), False)
])

ctl_gold_data = [
    ("gold_star_schema_pipeline", "dim_customer", datetime(2026, 1, 1, 0, 0, 0), datetime(2026, 1, 1, 0, 0, 0), "SUCCESS"),
    ("gold_star_schema_pipeline", "fact_orders", datetime(2026, 1, 1, 0, 0, 0), datetime(2026, 1, 1, 0, 0, 0), "SUCCESS")
]
spark.createDataFrame(ctl_gold_data, ctl_gold_schema).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ctl_gold_watermark")

# 2. Khởi tạo bảng Silver Customer Value (có cột _ingest_at để incremental load lên Gold)
cust_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("credit_limit", DoubleType(), True),
    StructField("updated_at", TimestampType(), True),     # Thời gian phát sinh thay đổi ở nguồn
    StructField("_ingest_at", TimestampType(), False)     # Thời gian nạp vào Silver Lakehouse (dùng làm Watermark cho Gold)
])
cust_data = [
    ("CUST_101", "Nguyen Van An", "Hai Phong", 30000000.0, datetime(2026, 1, 1, 10, 0, 0), datetime(2026, 1, 1, 10, 5, 0)),
    ("CUST_102", "Tran Thi Binh", "Danang", 20000000.0, datetime(2026, 1, 1, 8, 0, 0), datetime(2026, 1, 1, 8, 5, 0)),
    ("CUST_103", "Le Van Cuong", "Hanoi", 12000000.0, datetime(2026, 1, 1, 9, 30, 0), datetime(2026, 1, 1, 9, 35, 0))
]
spark.createDataFrame(cust_data, cust_schema).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_customer_value")

# 3. Khởi tạo bảng Silver Orders (Fact nguồn)
order_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("order_amount", DoubleType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_timestamp", TimestampType(), False),
    StructField("_ingested_at", TimestampType(), False)
])
order_data = [
    ("ORD_001", "CUST_101", 5500000.0, "COMPLETED", datetime(2026, 1, 1, 10, 15, 0), datetime(2026, 1, 1, 10, 20, 0)),
    ("ORD_002", "CUST_102", 3200000.0, "COMPLETED", datetime(2026, 1, 1, 8, 30, 0), datetime(2026, 1, 1, 8, 35, 0)),
    ("ORD_003", "CUST_999", 1200000.0, "PENDING", datetime(2026, 1, 1, 10, 45, 0), datetime(2026, 1, 1, 10, 50, 0))
]
spark.createDataFrame(order_data, order_schema).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_orders")

print("--- DỮ LIỆU TẦNG SILVER ĐÃ SẴN SÀNG (BẢNG VALUE CÓ CỘT _ingest_at) ---")
spark.table("silver_customer_value").show(truncate=False)
spark.table("silver_orders").show(truncate=False)


## BƯỚC 2: XÁC ĐỊNH MỐC LOAD TĂNG DẦN (INCREMENTAL WATERMARK FREEZE)
- **`start_watermark`**: Lấy từ `ctl_gold_watermark`.
- **`cutoff_watermark`**: Đóng băng tại thời điểm bắt đầu batch (`batch_start_time`).
- **Điều kiện lọc Incremental cho Bảng Value**: `_ingest_at > start_watermark AND _ingest_at <= cutoff_watermark`.

In [ ]:
batch_start_time = datetime.now()

# Lấy watermark của dim_customer
dim_watermark = spark.table("ctl_gold_watermark") \
    .filter("pipeline_name = 'gold_star_schema_pipeline' AND target_table = 'dim_customer'") \
    .select("last_watermark").collect()[0]["last_watermark"]

# Lấy watermark của fact_orders
fact_watermark = spark.table("ctl_gold_watermark") \
    .filter("pipeline_name = 'gold_star_schema_pipeline' AND target_table = 'fact_orders'") \
    .select("last_watermark").collect()[0]["last_watermark"]

print(f"[DIM]  Load tu {dim_watermark} den {batch_start_time}")
print(f"[FACT] Load tu {fact_watermark} den {batch_start_time}")

# Trích xuất dữ liệu tăng dần từ Silver dựa trên _ingest_at (Value) và _ingested_at (Orders)
df_inc_customers = spark.table("silver_customer_value") \
    .filter((col("_ingest_at") > lit(dim_watermark)) & (col("_ingest_at") <= lit(batch_start_time)))

df_inc_orders = spark.table("silver_orders") \
    .filter((col("_ingested_at") > lit(fact_watermark)) & (col("_ingested_at") <= lit(batch_start_time)))

print(f"Số bản ghi khách hàng cần cập nhật vào Gold (lọc theo _ingest_at): {df_inc_customers.count()}")
print(f"Số bản ghi đơn hàng mới cần nạp vào Gold: {df_inc_orders.count()}")


## BƯỚC 3: DỰNG STAR SCHEMA (DIMENSION & FACT CHUẨN KIMBALL)
1. **Surrogate Key:** Dùng `md5(customer_id)` tạo mã định danh duy nhất cho Dimension độc lập với mã nguồn.
2. **Unknown Member:** Xử lý các đơn hàng có `customer_id` mới chưa kịp có trong Dim bằng giá trị đại diện `md5('UNKNOWN')`.
3. **Date Key:** Chuyển đổi timestamp thành khóa ngày `YYYYMMDD` kiểu Integer để tăng tốc phép nối (JOIN).

In [ ]:
from delta.tables import DeltaTable

# 1. BIẾN ĐỔI BẢNG DIM_CUSTOMER (GOLD)
# Sử dụng _ingest_at làm effective_date cho Dimension
df_dim_customer_transformed = df_inc_customers \
    .withColumn("customer_dim_key", md5(col("customer_id"))) \
    .withColumn("effective_date", col("_ingest_at")) \
    .withColumn("_gold_inserted_at", current_timestamp()) \
    .select(
        "customer_dim_key",
        "customer_id",
        "customer_name",
        "address",
        "credit_limit",
        "effective_date",
        "_gold_inserted_at"
    )

# Ghi / Merge vào Delta Dim
if not spark.catalog.tableExists("dim_customer"):
    df_dim_customer_transformed.write.format("delta").mode("overwrite").saveAsTable("dim_customer")
else:
    tbl_dim = DeltaTable.forName(spark, "dim_customer")
    tbl_dim.alias("tgt").merge(
        df_dim_customer_transformed.alias("src"),
        "tgt.customer_id = src.customer_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# 2. BIẾN ĐỔI BẢNG FACT_ORDERS (GOLD)
# Lookup Surrogate Key từ dim_customer (Xử lý Unknown Member bằng LEFT JOIN)
df_all_dim = spark.table("dim_customer").select("customer_id", "customer_dim_key")

df_fact_orders_transformed = df_inc_orders.alias("ord") \
    .join(df_all_dim.alias("dim"), col("ord.customer_id") == col("dim.customer_id"), "left") \
    .withColumn("customer_dim_key", expr("coalesce(dim.customer_dim_key, md5('UNKNOWN'))")) \
    .withColumn("date_key", date_format(col("ord.order_timestamp"), "yyyyMMdd").cast(IntegerType())) \
    .withColumn("_gold_inserted_at", current_timestamp()) \
    .select(
        col("ord.order_id"),
        col("customer_dim_key"),
        col("date_key"),
        col("ord.order_amount"),
        col("ord.order_status"),
        col("ord.order_timestamp"),
        col("_gold_inserted_at")
    )

# Ghi / Merge vào Delta Fact
if not spark.catalog.tableExists("fact_orders"):
    df_fact_orders_transformed.write.format("delta").mode("overwrite").saveAsTable("fact_orders")
else:
    tbl_fact = DeltaTable.forName(spark, "fact_orders")
    tbl_fact.alias("tgt").merge(
        df_fact_orders_transformed.alias("src"),
        "tgt.order_id = src.order_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print("--- BẢNG GOLD DIM_CUSTOMER TRONG LAKEHOUSE ---")
spark.table("dim_customer").show(truncate=False)
print("--- BẢNG GOLD FACT_ORDERS TRONG LAKEHOUSE ---")
spark.table("fact_orders").show(truncate=False)


## BƯỚC 4: GIẢI MÃ CƠ CHẾ "MERGE INTO" TRÊN CLICKHOUSE & CÁCH CODE PYSPARK

### 1. Tại sao dbt lại dịch ra được lệnh MERGE INTO còn ClickHouse thì không có?
- **Cần phân biệt rõ Target Database:**
  - Nếu dbt của bạn trỏ vào **Fabric Synapse Warehouse (T-SQL)**: dbt sẽ dịch model incremental thành lệnh `MERGE INTO target USING source ON target.id = source.id` vì Microsoft SQL/Fabric hỗ trợ chuẩn này.
  - Nếu dbt trỏ vào **ClickHouse (`dbt-clickhouse`)**: ClickHouse bản chất **KHÔNG CÓ cú pháp ANSI SQL `MERGE INTO`**! Thay vào đó, `dbt-clickhouse` ngầm tạo ra một bảng tạm staging, rồi thực thi 2 lệnh:
    1. `ALTER TABLE target DELETE WHERE id IN (SELECT id FROM staging_table);`
    2. `INSERT INTO target SELECT * FROM staging_table;`

### 2. Hai cách thức thực hiện Upsert / Merge Into từ PySpark vào ClickHouse:
1. **Cách 1: Mô phỏng 100% cơ chế của dbt (Staging Table + Delete + Insert)**: Đảm bảo dữ liệu cũ bị xóa ngay lập tức trước khi chèn mới.
2. **Cách 2: Chuẩn tối ưu của ClickHouse (`ReplacingMergeTree`)**: ClickHouse tự deduplicate các bản ghi trùng key trong background, truy vấn dùng từ khóa `FINAL`.

In [ ]:
# ==============================================================================
# CÁCH 1: PYSPARK MÔ PHỎNG 100% CƠ CHẾ INCREMENTAL CỦA DBT TRÊN CLICKHOUSE
# Quy trình: Write DataFrame vào Staging Table -> Gửi lệnh SQL DELETE + INSERT
# ==============================================================================
def merge_into_clickhouse_like_dbt(
    df, 
    target_table, 
    unique_key, 
    host="localhost", 
    port=8123, 
    database="gold_analytics", 
    user="default", 
    password=""
):
    """
    Thực hiện tương đương MERGE INTO / Incremental dbt trên ClickHouse:
    Bước 1: PySpark ghi batch tăng dần vào bảng tạm (staging)
    Bước 2: ClickHouse xóa các bản ghi cũ trùng khóa (ALTER TABLE ... DELETE)
    Bước 3: ClickHouse chèn các bản ghi mới từ staging vào bảng chính
    Bước 4: Truncate bảng staging
    """
    staging_table = f"stg_{target_table}"
    jdbc_url = f"jdbc:ch://{host}:{port}/{database}"
    driver = "com.clickhouse.jdbc.ClickHouseDriver"
    
    print(f"\n[ClickHouse Upsert] Bắt đầu đồng bộ bảng {target_table} qua Staging Table...")
    
    # 1. Ghi đè dữ liệu batch mới vào bảng Staging trên ClickHouse
    df.write \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", staging_table) \
        .option("user", user) \
        .option("password", password) \
        .option("driver", driver) \
        .option("batchsize", "50000") \
        .mode("overwrite") \
        .save()
    print(f"  -> Đã nạp {df.count()} dòng vào bảng tạm {staging_table}")
    
    # 2. Kết nối JDBC thực thi lệnh Delete + Insert trên ClickHouse (giống hệt macro dbt)
    # Dùng DriverManager của Java có sẵn trong PySpark JVM
    jvm = spark._jvm
    jvm.java.lang.Class.forName(driver)
    conn = jvm.java.sql.DriverManager.getConnection(jdbc_url, user, password)
    statement = conn.createStatement()
    
    try:
        # Lệnh 1: Xóa các khóa cũ đã tồn tại trong bảng chính
        sql_delete = f"""
            ALTER TABLE {database}.{target_table} 
            DELETE WHERE {unique_key} IN (SELECT {unique_key} FROM {database}.{staging_table})
        """
        print(f"  -> Đang thực thi DELETE bản ghi trùng khóa theo {unique_key}...")
        statement.execute(sql_delete)
        
        # Lệnh 2: Insert toàn bộ dữ liệu mới từ staging vào bảng chính
        sql_insert = f"""
            INSERT INTO {database}.{target_table} 
            SELECT * FROM {database}.{staging_table}
        """
        print(f"  -> Đang thực thi INSERT dữ liệu mới vào {target_table}...")
        statement.execute(sql_insert)
        
        # Lệnh 3: Dọn dẹp bảng staging
        statement.execute(f"TRUNCATE TABLE {database}.{staging_table}")
        print(f"  [Thành công] MERGE INTO vào ClickHouse bảng {target_table} hoàn tất!")
    finally:
        statement.close()
        conn.close()

print("Đã định nghĩa hàm merge_into_clickhouse_like_dbt() sẵn sàng sử dụng!")


## BƯỚC 5: CẬP NHẬT WATERMARK TẦNG GOLD KHI HOÀN TẤT THÀNH CÔNG
Chỉ khi cả việc ghi vào Delta Gold (và ClickHouse) hoàn tất trọn vẹn, ta mới tiến hành cập nhật mốc Watermark mới để lần chạy tiếp theo không bị tải lại dữ liệu cũ.

In [ ]:
tbl_ctl_gold = DeltaTable.forName(spark, "ctl_gold_watermark")

df_new_watermarks = spark.createDataFrame([
    ("gold_star_schema_pipeline", "dim_customer", batch_start_time, datetime.now(), "SUCCESS"),
    ("gold_star_schema_pipeline", "fact_orders", batch_start_time, datetime.now(), "SUCCESS")
], ctl_gold_schema)

tbl_ctl_gold.alias("tgt").merge(
    df_new_watermarks.alias("src"),
    "tgt.pipeline_name = src.pipeline_name AND tgt.target_table = src.target_table"
).whenMatchedUpdate(set={
    "last_watermark": "src.last_watermark",
    "last_run_time": "src.last_run_time",
    "status": "src.status"
}).execute()

print("--- BẢNG WATERMARK TẦNG GOLD SAU KHI CẬP NHẬT THÀNH CÔNG ---")
spark.table("ctl_gold_watermark").show(truncate=False)


## BƯỚC 6: KIỂM CHỨNG THỰC NGHIỆM TÍNH BẤT BIẾN (IDEMPOTENCY) CỦA MERGE INTO

Phần này giúp bạn kiểm chứng thực tế trên Fabric Lakehouse qua 2 kịch bản:
1. **Kịch bản A (Replay 100% dữ liệu cũ):** Giả lập mạng bị đứt, scheduler kích hoạt chạy lại cùng 1 batch. Số dòng trong Star Schema **giữ nguyên 100%, tuyệt đối không bị duplicate**.
2. **Kịch bản B (Batch hỗn hợp - Vừa UPDATE vừa INSERT):** Có bản ghi cũ thay đổi thuộc tính (`CUST_101`, `ORD_001`) và có bản ghi mới hoàn toàn (`CUST_104`, `ORD_004`). Lệnh MERGE INTO sẽ tự động phân loại: bản ghi cũ được cập nhật tại chỗ, bản ghi mới được chèn thêm.

In [ ]:
# ==============================================================================
# KỊCH BẢN A: GIẢ LẬP CHẠY LẠI CHÍNH XÁC MẺ DỮ LIỆU CŨ (REPLAY TEST)
# ==============================================================================
print("=== [TRƯỚC KHI REPLAY] SỐ DÒNG HIỆN TẠI ===")
count_dim_before = spark.table("dim_customer").count()
count_fact_before = spark.table("fact_orders").count()
print(f"dim_customer: {count_dim_before} dòng")
print(f"fact_orders:   {count_fact_before} dòng")

print("\n--- Đang thực hiện chạy lại câu lệnh MERGE INTO với đúng batch dữ liệu cũ... ---")
tbl_dim = DeltaTable.forName(spark, "dim_customer")
tbl_dim.alias("tgt").merge(
    df_dim_customer_transformed.alias("src"),
    "tgt.customer_id = src.customer_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

tbl_fact = DeltaTable.forName(spark, "fact_orders")
tbl_fact.alias("tgt").merge(
    df_fact_orders_transformed.alias("src"),
    "tgt.order_id = src.order_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print("\n=== [SAU KHI REPLAY] KIỂM TRA SỐ DÒNG ===")
count_dim_after = spark.table("dim_customer").count()
count_fact_after = spark.table("fact_orders").count()
print(f"dim_customer: {count_dim_after} dòng (Trước: {count_dim_before}) -> {'KHÔNG BỊ TRÙNG (PASS)' if count_dim_before == count_dim_after else 'BỊ LỖI DUPLICATE'}")
print(f"fact_orders:   {count_fact_after} dòng (Trước: {count_fact_before}) -> {'KHÔNG BỊ TRÙNG (PASS)' if count_fact_before == count_fact_after else 'BỊ LỖI DUPLICATE'}")


In [ ]:
# ==============================================================================
# KỊCH BẢN B: MẺ DỮ LIỆU TIẾP THEO VỪA CÓ SỬA ĐỔI (UPDATE) VỪA CÓ THÊM MỚI (INSERT)
# - CUST_101: Sửa địa chỉ sang 'Da Nang', tăng hạn mức lên 50,000,000 (ĐÃ TỒN TẠI)
# - CUST_104: Khách hàng mới 'Pham Thi Dung' (CHƯA TỒN TẠI)
# - ORD_001:  Đổi trạng thái sang 'DELIVERED' (ĐÃ TỒN TẠI)
# - ORD_004:  Đơn hàng mới của CUST_104 giá trị 8,900,000 (CHƯA TỒN TẠI)
# ==============================================================================
batch_2_time = datetime(2026, 1, 1, 14, 0, 0)

# 1. Dữ liệu batch 2 của Silver Customer Value
new_cust_data = [
    ("CUST_101", "Nguyen Van An", "Da Nang", 50000000.0, datetime(2026, 1, 1, 13, 0, 0), batch_2_time), # UPDATE
    ("CUST_104", "Pham Thi Dung", "Can Tho", 45000000.0, datetime(2026, 1, 1, 13, 30, 0), batch_2_time) # INSERT
]
df_batch2_customers = spark.createDataFrame(new_cust_data, cust_schema)

# 2. Dữ liệu batch 2 của Silver Orders
new_order_data = [
    ("ORD_001", "CUST_101", 5500000.0, "DELIVERED", datetime(2026, 1, 1, 10, 15, 0), batch_2_time),     # UPDATE trạng thái
    ("ORD_004", "CUST_104", 8900000.0, "COMPLETED", datetime(2026, 1, 1, 13, 45, 0), batch_2_time)     # INSERT đơn mới
]
df_batch2_orders = spark.createDataFrame(new_order_data, order_schema)

# 3. Biến đổi Star Schema cho Batch 2
df_batch2_dim = df_batch2_customers \
    .withColumn("customer_dim_key", md5(col("customer_id"))) \
    .withColumn("effective_date", col("_ingest_at")) \
    .withColumn("_gold_inserted_at", current_timestamp()) \
    .select("customer_dim_key", "customer_id", "customer_name", "address", "credit_limit", "effective_date", "_gold_inserted_at")

# Merge Batch 2 vào dim_customer
tbl_dim.alias("tgt").merge(
    df_batch2_dim.alias("src"),
    "tgt.customer_id = src.customer_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# Biến đổi Fact cho Batch 2 (Lookup lại Dim)
df_all_dim_updated = spark.table("dim_customer").select("customer_id", "customer_dim_key")
df_batch2_fact = df_batch2_orders.alias("ord") \
    .join(df_all_dim_updated.alias("dim"), col("ord.customer_id") == col("dim.customer_id"), "left") \
    .withColumn("customer_dim_key", expr("coalesce(dim.customer_dim_key, md5('UNKNOWN'))")) \
    .withColumn("date_key", date_format(col("ord.order_timestamp"), "yyyyMMdd").cast(IntegerType())) \
    .withColumn("_gold_inserted_at", current_timestamp()) \
    .select(col("ord.order_id"), col("customer_dim_key"), col("date_key"), col("ord.order_amount"), col("ord.order_status"), col("ord.order_timestamp"), col("_gold_inserted_at"))

# Merge Batch 2 vào fact_orders
tbl_fact.alias("tgt").merge(
    df_batch2_fact.alias("src"),
    "tgt.order_id = src.order_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print("=== BẢNG DIM_CUSTOMER SAU KHI MERGE BATCH 2 (CUST_101 CẬP NHẬT, CUST_104 ĐƯỢC THÊM) ===")
spark.table("dim_customer").show(truncate=False)

print("=== BẢNG FACT_ORDERS SAU KHI MERGE BATCH 2 (ORD_001 THÀNH DELIVERED, ORD_004 ĐƯỢC THÊM) ===")
spark.table("fact_orders").show(truncate=False)


In [ ]:
# ==============================================================================
# KIỂM CHỨNG LỊCH SỬ GHI TRANSACTION LOG CỦA DELTA LAKE (DELTA TIME TRAVEL)
# ==============================================================================
print("--- LỊCH SỬ THAY ĐỔI CỦA BẢNG DIM_CUSTOMER (CÁC PHIÊN BẢN COMMIT) ---")
spark.sql("DESCRIBE HISTORY dim_customer") \
    .select("version", "timestamp", "operation", "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated") \
    .show(truncate=False)

print("--- LỊCH SỬ THAY ĐỔI CỦA BẢNG FACT_ORDERS (CÁC PHIÊN BẢN COMMIT) ---")
spark.sql("DESCRIBE HISTORY fact_orders") \
    .select("version", "timestamp", "operation", "operationMetrics.numTargetRowsInserted", "operationMetrics.numTargetRowsUpdated") \
    .show(truncate=False)
